# Ensemble Member Generation for SVS Model Simulation of E1 Enclosure

This script creates an ensemble of parameter sets to be used for simulating the E1 enclosure with the SVS model. The ensemble is generated using Latin Hypercube Sampling (LHS) to efficiently explore the parameter space.

## Methodology

1. **Parameter Definition:**
   - Specifies the soil hydraulic parameters to be perturbed in the ensemble:
     - `wsat` (VWC at saturation)
     - `wfc` (VWC at field capacity)
     - `wwilt` (VWC at wilting point)
     - `wunfrz` (residual VWC at freezing)
     - `ksat` (saturated hydraulic conductivity)
     - `psisat` (BC_psi_ae, air entry pressure)
     - `bcoef` (BC_b, Brooks-Corey b-coefficient)
   - Defines the mapping between full parameter names and their corresponding names in the SVS model.

2. **Range Definition:**
   - Specifies the ranges for each parameter, separately for topsoil and cover material layers.
   - These ranges are represented as uniform distributions using `scipy.stats.uniform`.

3. **Latin Hypercube Sampling:**
   - Generates a set of `N_ENS` samples using LHS to efficiently cover the parameter space.
   - `pyDOE` library is used for LHS implementation.

4. **Ensemble Generation:**
   - Initializes a dictionary (`scenarios`) to store the ensemble members.
   - For each ensemble member:
     - Assigns initial NaN values to all parameters for all soil layers.
     - Maps LHS samples to the defined parameter ranges for topsoil and cover material layers.
     - Calculates `wwilt` based on `wsat`, `psisat`, and `bcoef` using the Brooks-Corey model.
     - Sets `user_wfcdp` based on the last layer's `wsat`.

5. **Data Saving:**
   - Saves the generated `scenarios` dictionary as a pickle file for later use in SVS simulations.

## File Paths

* **Output File:**
    - `./Results/30scenarios_5params_Mar13.pickle`

## Parameters

- `N_ENS`: Number of ensemble members (default: 30).
- `RSEED`: Random seed for reproducibility (default: 1915).

## Dependencies

- **Python libraries:** `pathlib`, `pickle`, `pandas`, `numpy`, `scipy.stats`, `pyDOE`
- **Helper function:** `map_to_range`, `swrc_model` (assumed to be defined in a separate `helpers` module)


## Author

Alireza Amani

## Date

2024-07-23

In [1]:
'''
Here we create the ensemble members for the SVS model to simulate the
E1 enclosure.

Steps:
    1. Identify the parameters that we want to perturb
    2. Gather information about the parameters
    3. Create the ensemble members
    4. Save the ensemble members

'''
# Import modules --------------------------------------------------------------
from pathlib import Path
from importlib import reload
import pickle

import pandas as pd
import numpy as np
from scipy.stats import uniform, lognorm
from pyDOE import lhs

from helpers import map_to_range, swrc_model
# ______________________________________________________________________________
# Variables --------------------------------------------------------------------

# relevant paths for this notebook
paths = dict(
    save_scen_dir = Path(
        "./Results/"
    ),
)
LOCAL_TZ = "America/Montreal"

N_ENS = 30 # number of ensemble members

RSEED = 1915 # random seed

# checks
for k, v in paths.items():
    assert v.exists(), f"{k} does not exist: {v}"

# ______________________________________________________________________________
# Parameter information --------------------------------------------------------

# 1. parameters to perturb
# full param name: param name in the model
perb_params = {

    # soil texture, %
    # "sand percentage":                                      "sand",
    # "clay percentage":                                      "clay",

    # volumetric water content, VWC (m3/m3)
    "VWC at saturation":                                    "wsat",
    "VWC at field capacity":                                "wfc",
    "VWC at wilting point":                                 "wwilt",
    "VWC at field capacity":                                "wfc",
    "residual VWC at freezing":                             "wunfrz",

    # vertical saturated hydraulic conductivity
    "saturated hydraulic conductivity":                     "ksat", # (m/s)

    # SWRC parameters - Brooks-Corey
    "BC_psi_ae":                                            "psisat", # (mH2O)
    "BC_b":                                                 "bcoef", # no units
}

LHS_samples = lhs(len(perb_params), samples=N_ENS)


'''
Topsoil Ranges:

bcoef: 1.0 ~ 2.0
psisat: 0.05 ~ 0.45
ksat: 5.e-07 ~ 5.e-06
wsat: 0.39 ~ 0.43
wfc: 0.08 ~ 0.17
psi (wfc): 1.0 ~ 1.6 mh2o

'''

# E1 enclosure consists of two soil types: Topsoil (TS) and Cover Material (CM)
# we need to define ranges for direct perturbation of parameters for each soil type
topsoil_direct_perb_range = {
    # "sand percentage":                    uniform(loc=60.0, scale=5.0),
    # "clay percentage":                    uniform(loc=5.0, scale=3.0),
    "VWC at saturation":                  uniform(loc=0.39, scale=0.04),
    "VWC at field capacity":              uniform(loc=0.08, scale=0.09),
    "residual VWC at freezing":           uniform(loc=0.06, scale=0.04),
    "saturated hydraulic conductivity":   uniform(loc=5.0E-07, scale=4.5E-06),
    "BC_psi_ae":                          uniform(loc=0.05, scale=0.4),
    "BC_b":                               uniform(loc=1.0, scale=1.0),
}

'''
Cover Ranges:
- `wsat_cover`: 0.35 - 0.37
- `wfc_cover`: 0.22 - 0.28 (0.20 ~ 0.28, inf by midcover) (1.0 ~ 1.6 mh2o)
- `ksat_cover`: 1.0e-06 - 3.0e-06 (5.0e-06, inf by midcover)
- `bcoef_cover`: 1.0 - 3.5
- `psisat_cover`: 0.6 - 0.8

'''

covermat_direct_perb_range = {
    "VWC at saturation":                 uniform(loc=0.35, scale=0.02),
    "VWC at field capacity":             uniform(loc=0.20, scale=0.08),
    "residual VWC at freezing":          uniform(loc=0.03, scale=0.02),
    "saturated hydraulic conductivity":  uniform(loc=1.0E-06, scale=4.0E-06),
    "BC_psi_ae":                         uniform(loc=0.6, scale=0.2),
    "BC_b":                              uniform(loc=1.0, scale=2.5),
}


# enc layering
# 15 cm Topsoil + 175 cm Cover Material
# assume all layers are 5 cm thick
# e1_layering = "15:5:Topsoil + 175:5:Cover Material"
e1_layering = "15:2.5:Topsoil + 160:5:CoverMaterial + 15:2.5:CoverMaterial"

topsoil_indices = list(range(0, 6))
covermat_indices = list(range(6, 44))
NLAYERS = len(topsoil_indices) + len(covermat_indices)

# precision dict
precision_params = {
    # "sand":                             1,
    # "clay":                             1,
    "wsat":                             3,
    "wfc":                              3,
    "wwilt":                            3,
    "wfc":                              3,
    "wunfrz":                           3,
    "user_wfcdp":                       3,
    "ksat":                             8,
    "psisat":                           3,
    "bcoef":                            3,
}
# ______________________________________________________________________________
# Create ensemble members ------------------------------------------------------

scenarios = dict()

for icntr in range(N_ENS):
    scenarios[F"scen_{icntr}"] = dict()

    for values in perb_params.values():
        scenarios[F"scen_{icntr}"][values] = np.full(NLAYERS, np.nan)

    # scenarios[F"scen_{icntr}"]["SCP"] = np.nan # snow conductivity power term
    scenarios[F"scen_{icntr}"]["user_wfcdp"] = np.nan # snow conductivity power term

# set random seed
np.random.seed(RSEED)

# Iterate over scenarios
for icntr in range(N_ENS):
    scen_name = F"scen_{icntr}"

    # Mapping LHS samples to each parameter range
    for idx, (fullname, codename) in enumerate(perb_params.items()):
        if codename not in ["wwilt"]:

            param_range_topsoil = topsoil_direct_perb_range[fullname]
            param_range_covermat = covermat_direct_perb_range[fullname]

            sample_value_topsoil = map_to_range(LHS_samples[icntr, idx], param_range_topsoil)
            sample_value_covermat = map_to_range(LHS_samples[icntr, idx], param_range_covermat)

            scenarios[scen_name][codename][topsoil_indices] = np.round(
                np.full(len(topsoil_indices), sample_value_topsoil),
                precision_params[codename]
            )

            scenarios[scen_name][codename][covermat_indices] = np.round(
                np.full(len(covermat_indices), sample_value_covermat),
                precision_params[codename]
            )


    scenarios[scen_name]["wwilt"] = np.round(
        swrc_model(
            psi_mh2o=1500 * 0.102,
            theta_sat=scenarios[scen_name]["wsat"],
            psi_ae=scenarios[scen_name]["psisat"],
            bcoef=scenarios[scen_name]["bcoef"]
        ),
        precision_params["wwilt"]
    )


    scenarios[F"scen_{icntr}"]["user_wfcdp"] = np.round(
        scenarios[scen_name]["wsat"][-1] - 0.005,
        precision_params["user_wfcdp"]
    )

# save the scenarios
with open(paths["save_scen_dir"] / "30scenarios_5params_Mar13.pickle", "wb") as f:
    pickle.dump(scenarios, f)

print("Done!")
# ______________________________________________________________________________

Done!
